### Converting to Tensor
 In TensorFlow 2.x, the framework is designed to be extremely "NumPy-friendly." When you pass a NumPy array into a TensorFlow function (like tf.data.Dataset.from_tensor_slices), TensorFlow automatically converts it into a tf.Tensor under the hood.


In [ ]:
import tensorflow as tf
import numpy as np

arr1 = np.array([1,2,3,4,5]).reshape(5,1)
immutable  = tf.constant(arr1, dtype = tf.float32)    # values fixed and cant be changed
mutable = tf.Variable(arr1, dtype = tf.float32)        # values can be changed during training for example parameters need to be updated during training 
t1 = tf.convert_to_tensor(arr1, dtype=tf.float32)
print(t1)
t1 = tf.reshape(t1,[-1])  # tensorflow does not provide any direct flatten() fuction as torch or numpy but we can either use reshape or tf.keras.layers.Flatten() can be used !!! 
print(t1.shape)
t1

tf.Tensor(
[[1.]
 [2.]
 [3.]
 [4.]
 [5.]], shape=(5, 1), dtype=float32)
(5,)


<tf.Tensor: shape=(5,), dtype=float32, numpy=array([1., 2., 3., 4., 5.], dtype=float32)>

### Data Pipelines

In [ ]:
features  = np.random.sample((100, 8))
labels = np.random.sample((100, 1))

dataset = tf.data.Dataset.from_tensor_slices((features, labels))

dataset = dataset.shuffle(buffer_size=100).batch(32).prefetch(tf.data.AUTOTUNE) # .prefetch(tf.data.AUTOTUNE). This tells TensorFlow to prepare the next batch of data while the GPU is still working on the current one, eliminating bottlenecks.

print(f'Tensorflow Pipeline Ready')

Tensorflow Pipeline Ready


### Model

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers

model_tf = Sequential([
    layers.Dense(16, activation = 'relu', input_shape = (8, )),  # A Dense layer in TensorFlow (tf.keras.layers.Dense) is a fully connected neural network layer where every neuron is connected to every neuron in the previous layer. It performs a weighted sum of inputs plus a bias, followed by an optional activation function.
    layers.Dense(1) # automatically infers output shape of previous layer as current layer's input 
]) # also only entry point needs the input shape in tensorflow

model_tf.compile(optimizer = 'adam', loss = 'mse')

### Training Mechanics

In [ ]:
history = model_tf.fit(
    dataset,
    epochs = 10,
    callbacks = [tf.keras.callbacks.TensorBoard(log_dir='./logs')]
)  
# To use Tensorboard use " tensorboard --logdir=logs " in the terminal

Epoch 1/10
4/4 [==============================] - 2s 10ms/step - loss: 0.1792
Epoch 2/10
4/4 [==============================] - 0s 7ms/step - loss: 0.1728
Epoch 3/10
4/4 [==============================] - 0s 7ms/step - loss: 0.1703
Epoch 4/10
4/4 [==============================] - 0s 3ms/step - loss: 0.1680
Epoch 5/10
4/4 [==============================] - 0s 5ms/step - loss: 0.1653
Epoch 6/10
4/4 [==============================] - 0s 4ms/step - loss: 0.1632
Epoch 7/10
4/4 [==============================] - 0s 6ms/step - loss: 0.1607
Epoch 8/10
4/4 [==============================] - 0s 6ms/step - loss: 0.1584
Epoch 9/10
4/4 [==============================] - 0s 7ms/step - loss: 0.1563
Epoch 10/10
4/4 [==============================] - 0s 6ms/step - loss: 0.1540


In [ ]:
# Using Early Stopping
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience= 3,
    restore_best_weights=True
)
history = model_tf.fit(dataset,
            epochs = 10,
            callbacks = [early_stop])

In [4]:
#Example of flatten() function
Y_test = np.random.random((10,1))
print("Before Flattening",Y_test.shape)
print("After Flattening",Y_test.flatten().shape)

Before Flattening (10, 1)
After Flattening (10,)


## SimpleAttention Layer – Mathematical Breakdown

### Step 1: Inputs

Suppose your input is a batch of sequences:

$$
X \in \mathbb{R}^{B \times T \times U}
$$

- $B$: batch size  
- $T$: sequence length (number of time steps)  
- $U$: feature dimension (units per step)

---

### Step 2: Learnable weight vector

$$
w \in \mathbb{R}^{U \times 1}
$$

This is a trainable parameter vector.

---

### Step 3: Compute scores

You perform a matrix multiplication:

$$
S = X \cdot w
$$

Shape: $(B \times T \times 1)$.  
This gives a scalar score for each time step in the sequence.

---

### Step 4: Softmax normalization

You normalize across the time dimension:

$$
\alpha_t = \frac{e^{S_t}}{\sum_{j=1}^{T} e^{S_j}}
$$

So the weights $\alpha \in \mathbb{R}^{B \times T \times 1}$ form a probability distribution over time steps.

---

### Step 5: Apply weights

Finally, you scale the input features by these attention weights:

$$
Y_t = X_t \cdot \alpha_t
$$

So the output has the same shape as the input $(B \times T \times U)$, but each time step’s features are reweighted according to its learned importance.

---

### Intuition

This is a very simplified attention mechanism:  
- Instead of computing query–key dot products, it uses a single learned vector $w$ to score each time step.  
- The softmax ensures the scores become a distribution across time steps.  
- Multiplying back by the weights emphasizes “important” time steps and suppresses less relevant ones.
- This is a very simplified attention mechanism: instead of computing query–key dot products, it uses a single learned vector 𝑤 to score each time step.

In [ ]:
class SimpleAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(SimpleAttention, self).__init__()
        self.w = self.add_weight(shape = (units, 1), initializer = "random_normal")

    def call(self, inputs):
        scores = tf.matmul(inputs, self.w)
        weights = tf.nn.softmax(scores, axis = 1)
        return inputs * weights

# The Identity Layer: A Conceptual Guide

An **Identity Layer** acts as a *pass-through* or a *bridge* within a neural network.  
It takes an input and returns that exact same input as the output, without modifying it or applying any mathematical operations like weights or biases.

---

## 🛠️ Why Do We Use It?

### 1. Residual (Skip) Connections
In deep networks like **ResNet**, identity layers allow the signal to *skip* certain blocks.  
This helps prevent gradients from vanishing during backpropagation by providing a direct path for the gradient to flow through the network.

### 2. Placeholder for Research
When prototyping a new architecture, you can use an identity layer as a temporary placeholder for a complex module or a custom layer that you haven't built or finalized yet.

### 3. Architecture Consistency
Sometimes a specific network architecture requires a layer to exist to keep the code structure or *shape* consistent (for example, in conditional branching), even if that specific branch shouldn't actually change the data.


In [1]:
import tensorflow as tf

class IdentityLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(IdentityLayer, self).__init__()

    def call(self, input):
        # return input as it is 
        return input

**No such direct Identity Layer exists in keras.layers as there is for torch.nn**

# Sequence Models

#### In sequence modeling, we usually deal with a 3D Tensor: **(Batch Size, Time Steps, Features).**


#### **TensorFlow**: By default, it expects (batch, timesteps, features). You define this using the input_shape=(timesteps, features) argument in the first layer.

In [ ]:
import tensorflow as tf

# Data shape: [Batch, Time_Steps, Features]
model_lstm_tf = tf.keras.Sequential([
    # return_sequences=True is needed if another LSTM layer follows
    tf.keras.layers.LSTM(64, return_sequences=True, input_shape=(10, 5)),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(1)
])

# The Output Mystery: Hidden vs. Cell States

An **LSTM** has two types of "memory":
- **Hidden State (\(h\))** → short-term memory  
- **Cell State (\(c\))** → long-term memory  

---

##  Feature Comparison: TensorFlow vs. PyTorch

| Feature            | TensorFlow (`layers.LSTM`)                                                                 | PyTorch (`nn.LSTM`)                                                                 |
|--------------------|---------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------|
| **Standard Output** | Returns only the hidden state of the last step (or all steps if `return_sequences=True`).   | Returns both the full sequence of hidden states **AND** a tuple containing the final \((h, c)\). |
| **Getting States**  | You must explicitly set `return_state=True` to get the \((h, c)\).                          | It always returns the final states by default.                                      |

---

###  Key Takeaway
- **TensorFlow** is more minimal by default — you only get hidden states unless you ask for cell states.  
- **PyTorch** is more verbose — it always gives you both hidden and cell states, plus the full sequence of hidden states.


# Many to Many and Many to One Architecture

In [ ]:
# Many-to-Many RNN
model_many_many = tf.keras.Sequential([
    # return_sequences=True ensures every time step produces an output
    tf.keras.layers.SimpleRNN(64, return_sequences=True, input_shape=(10, 5))
])

# Many-to-One LSTM (with explicit states)
lstm_layer = tf.keras.layers.LSTM(64, return_state=True)
# output is the last hidden state; h and c are the final states
output, h, c = lstm_layer(input_data)

TensorFlow only returns (h, c) if you ask with return_state=True. Otherwise, you just get hidden states.

In [ ]:
import tensorflow as tf

x = tf.random.normal((32, 10, 8))  # (batch_size=32, seq_len=10, features=8)

# return_sequences=True gives all hidden states
lstm = tf.keras.layers.LSTM(16, return_sequences=True)
output = lstm(x)

print(output.shape)  # (32, 10, 16)

# last time step hidden state
last_output = output[:, -1, :]
print(last_output.shape)  # (32, 16)

# if you want both h and c, set return_state=True
lstm = tf.keras.layers.LSTM(16, return_sequences=True, return_state=True)
output, h, c = lstm(x)

print(h.shape)  # (32, 16) -> final hidden state
print(c.shape)  # (32, 16) -> final cell state
